In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Base URLs
STATE_URL = "https://www.homelessshelterdirectory.org/state/ohio"  # Change state here
BASE_URL = "https://www.homelessshelterdirectory.org"
MAX_ITERATIONS = 30  # Stop after 30 city pages

def get_city_links(state_url):
    """Extract all city URLs from the state page."""
    city_links = []
    response = requests.get(state_url)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        city_elements = soup.select("td a")  # City links are inside <td> tags
        
        for city in city_elements:
            link = city.get("href")
            if link and "/city/" in link:
                # Ensure absolute URL
                if not link.startswith("http"):
                    link = BASE_URL + link
                city_links.append(link)
    
    return city_links

def get_shelter_names(city_url):
    """Extract only real shelter names from a city page."""
    shelters = []
    response = requests.get(city_url)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Look for shelters only inside "layout_post_2 clearfix" sections
        shelter_sections = soup.select("div.layout_post_2.clearfix")

        for section in shelter_sections:
            shelter_name_tag = section.select_one("div.item_content h4")
            if shelter_name_tag:
                shelter_name = shelter_name_tag.text.strip()
                shelters.append(shelter_name)

    return shelters

def scrape_state_shelters(state_url, max_iterations=MAX_ITERATIONS):
    """Scrape up to `max_iterations` shelters for a given state."""
    city_links = get_city_links(state_url)
    all_shelters = []
    total_shelters = 0

    for i, city_link in enumerate(city_links):
        if i >= max_iterations:
            print(f"Reached {max_iterations} iterations. Stopping.")
            break

        print(f"\nScraping City {i+1}/{max_iterations}: {city_link}")
        shelters = get_shelter_names(city_link)

        if not shelters:
            print("  → No shelters found for this city.")

        for shelter in shelters:
            total_shelters += 1
            print(f"  🏠 {total_shelters}. Shelter Found: {shelter}")
            all_shelters.append({"City URL": city_link, "Shelter Name": shelter})
        
        time.sleep(1)  # Add delay to prevent getting blocked
    
    return all_shelters

# Run script for the given state (limited to 30 iterations)
shelter_data = scrape_state_shelters(STATE_URL, MAX_ITERATIONS)

# Print final summary
print(f"\nTotal Shelters Scraped: {len(shelter_data)}")

# Save to CSV
df = pd.DataFrame(shelter_data)
df.to_csv("homeless_shelters_ohio_limited.csv", index=False)

print("\n✅ Scraping completed. Data saved to 'homeless_shelters_ohio_limited.csv'.")


Scraping City 1/30: https://www.homelessshelterdirectory.org/city/oh-aberdeen
  🏠 1. Shelter Found: Adams County Ohio Shelter for the Homeless

Scraping City 2/30: https://www.homelessshelterdirectory.org/city/oh-aberdeen_village
  🏠 2. Shelter Found: Winter Sanctuary Homeless Shelter

Scraping City 3/30: https://www.homelessshelterdirectory.org/city/oh-ada
  🏠 3. Shelter Found: New Hope Ministry Park Shelter
  🏠 4. Shelter Found: Lima Rescue Mission
  🏠 5. Shelter Found: Family Promise Lima-Allen County

Scraping City 4/30: https://www.homelessshelterdirectory.org/city/oh-ada_village
  🏠 6. Shelter Found: Winter Sanctuary Homeless Shelter

Scraping City 5/30: https://www.homelessshelterdirectory.org/city/oh-adams_township
  🏠 7. Shelter Found: Winter Sanctuary Homeless Shelter

Scraping City 6/30: https://www.homelessshelterdirectory.org/city/oh-adamsville
  🏠 8. Shelter Found: Salvation Army Zanesville
  🏠 9. Shelter Found: Trulight Ministries

Scraping City 7/30: https://www.homele